## 0. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add langchain-openai langchain-community langchain-text-splitters faiss-cpu python-dotenv

### 2) 환경 변수와 설정

- `.env` 파일에 `OPENAI_API_KEY` 등록 (키 값은 출력하지 않고 설정 여부만 확인)

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (OpenAIEmbeddings, ChatOpenAI 가 OPENAI_API_KEY 환경 변수를 자동으로 읽음)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


In [2]:
# 임베딩 모델: 기존 코드(OpenAIEmbeddings())의 기본값을 명시 (1536차원)
EMBEDDING_MODEL_NAME = "text-embedding-ada-002"
print(EMBEDDING_MODEL_NAME)

CHAT_MODEL_NAME = "gpt-4o-mini"
print(CHAT_MODEL_NAME)

# FAISS 인덱스(벡터 DB) 저장 경로
# 경로 구분자는 슬래시(/)를 사용: "..\db\faiss..." 처럼 역슬래시를 쓰면 \f 가 이스케이프 문자로 해석되어 오류 발생
# Ollama 버전(faiss_index_ollama)과 임베딩 모델이 달라 인덱스가 호환되지 않으므로 경로를 분리
DB_PATH = "../db/faiss_index_openai"
print(DB_PATH)

# version1(개선 버전) 인덱스 저장 경로: 기본 인덱스(DB_PATH)를 덮어쓰지 않도록 별도 경로 사용
DB_PATH_V1 = "../db/faiss_index_openai_v1"
print(DB_PATH_V1)

text-embedding-ada-002
gpt-4o-mini
../db/faiss_index_openai
../db/faiss_index_openai_v1


#### RAG 파이프 라인
* Load Data - Text Split - Indexing - Retrieval - Generation
* OpenAIEmbeddings 사용

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from pprint import pprint

# 1. Load Data
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

print(type(documents), len(documents)) #[Document]
print(type(documents[0]))
print(documents[0])

C:\Users\user\AppData\Local\Temp\ipykernel_12568\867253867.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


<class 'list'> 1
<class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.
    바. 대통령령으로 정하는 작물재배업에서 발생하는 소득
    사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발

In [4]:
# 2️. Text Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(documents)

print(len(split_docs), type(split_docs), type(split_docs[0]))
print(split_docs[0])
print('두번째 Document ====================')
print(split_docs[1])

12 <class 'list'> <class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득' metadata={'source': '../data/taxinfo.txt'}
두번째 Document ====================
page_content='가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서

In [5]:
# 3️. Indexing (벡터 저장)
embeddings_model = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)
# vector DB에 저장
vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings_model
)

# 로컬 파일로 저장
vectorstore.save_local(DB_PATH)
print(f"✓ FAISS 인덱스 저장 완료: {DB_PATH}")

✓ FAISS 인덱스 저장 완료: ../db/faiss_index_openai


In [6]:
# 4️. Retrieval (유사 문서 검색) k: 질의와 가장 유사한 문서(청크) 6개를 찾아 반환하기
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
# retriever변수는 VectoreStoreRetriever 객체이다.
print(type(retriever))
# **질문(쿼리)**에 대해 유사한 문서를 검색하는 역할
retrieved_docs = retriever.invoke("소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?")
print(type(retrieved_docs), len(retrieved_docs))
print(type(retrieved_docs[0]))
print(retrieved_docs[0])

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
<class 'list'> 6
<class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득' metadata={'source': '../data/taxinfo.txt'}


In [7]:
# 5️. Generation (LLM 응답 생성)
# temperature=0: 같은 입력이면 같은 결과가 나오도록 (비교 실험의 재현성)
llm = ChatOpenAI(model=CHAT_MODEL_NAME, temperature=0)

# 유사도 검색의 결과를 str 문자열로 만들기
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
print(context)

제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득

가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.
    바. 대통령령으로 정하는 작물재배업에서 발생하는 소득

마. 「고용보험법」에 따라 받는 실업급여, 육아휴직 급여, 육아기 근로시간 단축 급여, 출산전후휴가 급여등, 「제대군인 지원에 관한 법률」에 

#### context 유무 비교 (RAG 효과 확인)

같은 질문을 **검색된 context를 주었을 때**와 **주지 않았을 때**로 나누어 답변을 비교합니다. 공정한 비교가 되려면 **context 유무만 다르고** 나머지 조건은 같아야 합니다.

| 조건 | 두 실험에서 |
|---|---|
| 모델, 생성 설정(`llm`) | 같음 (`llm` 하나를 공유, `temperature=0`) |
| 질문(`QUESTION`) | 같음 |
| 프롬프트 형식 | 같은 함수 `ask()`를 사용, `[관련 정보]` 부분만 있고 없고 |
| **context** | **다름 (이 차이만 비교)** |

> 두 결과를 `data/taxinfo.txt`의 **제12조 원문과 비교**하여, 어느 답변이 원문에 충실한지 직접 확인하세요.

In [8]:
QUESTION = "소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?"


def ask(question: str, context: str | None = None) -> str:
    """context 유무만 다르고 나머지 조건(모델, 질문)은 같은 질의 함수"""
    if context:
        prompt = f"다음 관련 정보를 근거로 질문에 답하세요.\n\n[관련 정보]\n{context}\n\n[질문]\n{question}"
    else:
        prompt = f"질문에 답하세요.\n\n[질문]\n{question}"
    return llm.invoke(prompt).content

In [9]:
print('context 적용한 결과')
pprint(ask(QUESTION, context))

context 적용한 결과
('소득세법에서 비과세소득에 해당하는 소득은 다음과 같습니다:\n'
 '\n'
 '1. **공익신탁법에 따른 공익신탁의 이익**\n'
 '2. **사업소득 중 다음 각 목의 소득**:\n'
 '   - 가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n'
 '   - 나. 1개의 주택을 소유하는 자의 주택임대소득 (기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 '
 '제외)\n'
 '   - 다. 대통령령으로 정하는 농어가부업소득\n'
 '   - 라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득\n'
 '   - 마. 조림기간 5년 이상인 임지의 임목의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액\n'
 '   - 바. 대통령령으로 정하는 작물재배업에서 발생하는 소득\n'
 '   - 사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발생하는 소득\n'
 '\n'
 '3. **근로소득과 퇴직소득 중 다음 각 목의 소득**:\n'
 '   - 가. 대통령령으로 정하는 복무 중인 병이 받는 급여\n'
 '   - 나. 법률에 따라 동원된 사람이 그 동원 직장에서 받는 급여\n'
 '   - 다. 산업재해보상보험법에 따라 수급권자가 받는 각종 급여\n'
 '   - 라. 근로기준법 또는 선원법에 따라 근로자 및 그 유족이 받는 각종 보상금\n'
 '\n'
 '4. **종교인소득 중 다음의 소득**:\n'
 '   - 종교관련종사자가 받는 대통령령으로 정하는 학자금, 식사 또는 식사대, 실비변상적 성질의 지급액, 출산이나 6세 이하 자녀의 '
 '보육과 관련하여 받는 금액 (월 20만원 이내), 사택을 제공받아 얻는 이익\n'
 '\n'
 '5. **기타소득 중 다음 각 목의 소득**:\n'
 '   - 국가유공자 등 예우 및 지원에 관한 법률에 따라 받는 보훈급여금, 학습보조비 및 북한이탈주민의 보호 및 정착지원에 관한 법률에 '
 '따라 받는 정착금, 보로금\n'
 ' 

In [10]:
print('context 적용하지 않은 결과')
pprint(ask(QUESTION))

context 적용하지 않은 결과
('소득세법에서 비과세소득에 해당하는 소득은 법에서 정한 특정 조건을 충족하는 소득으로, 세금이 부과되지 않는 소득을 의미합니다. 일반적으로 '
 '비과세소득에는 다음과 같은 항목들이 포함됩니다:\n'
 '\n'
 '1. **상속세 및 증여세**: 상속이나 증여로 받은 재산은 비과세 소득으로 간주됩니다.\n'
 '2. **보험금**: 생명보험이나 손해보험에서 지급받는 보험금은 비과세 소득입니다.\n'
 '3. **기타 비과세 소득**: 특정 조건을 충족하는 장학금, 보조금, 공공기관에서 지급하는 특정 지원금 등도 비과세 소득으로 인정될 수 '
 '있습니다.\n'
 '\n'
 '각 항목에 대한 구체적인 조건이나 한도는 소득세법에 명시되어 있으므로, 자세한 사항은 관련 법령을 참고하는 것이 좋습니다.')


### 개선한 Source - version1
* Retriever 검색방법 개선
    * search_type="mmr",  # 최대 다양성 검색
    * search_kwargs={"k": 3, "fetch_k": 6}  

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings

from pprint import pprint

# 1. 데이터 로드 (기존과 동일)
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

# 2. 텍스트 분할 개선
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 크기 증가
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],  # 자연스러운 분할을 위한 구분자
    length_function=len,
    is_separator_regex=False,
)
split_docs = splitter.split_documents(documents)

# 3. 인덱싱 (벡터 저장)
embeddings_model = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)

vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings_model
)

vectorstore.save_local(DB_PATH_V1)
print(f"✓ FAISS 인덱스 저장 완료: {DB_PATH_V1}")

# 4. 검색 개선
"""
    최대 다양성 검색(Maximum Marginal Relevance, MMR)
    MMR은 유사도가 높은 문서를 찾는 것을 넘어, 유사도와 다양성이라는 두 가지 기준을 모두 고려함
    - search_type="mmr": 검색 방식을 MMR로 지정합니다.
    - fetch_k: 일차적으로 질의와 유사한 문서 6개를 벡터 저장소에서 가져옵니다.
    - k: fetch_k로 가져온 6개의 문서 중에서 최종적으로 3개를 선택합니다. 3개를 선택할 때, MMR 알고리즘은 다음 두 가지를 고려함
        : 질의와의 유사도가 높고, 이미 선택된 다른 문서들과의 유사도가 낮은 (즉, 내용이 다양한) 문서
    * MMR의 작동 원리:
    - 질의와 가장 유사한 fetch_k개(6개)의 문서를 예비 후보군으로 가져옵니다.
    - 이 후보군 중에서 질의와 가장 유사한 문서 하나를 첫 번째 결과로 선택합니다.
    - 남은 후보군 중에서 질의와의 유사도는 높으면서 (관련성),
      이미 선택된 문서들과의 유사도는 낮은 (다양성) 문서를 찾아 다음 결과로 추가함
    - 이 과정을 k개(3개)의 문서가 모두 선택될 때까지 반복합니다.
    * 주의: 이 문서는 청크가 7개뿐이라 k=6, fetch_k=10 이면 거의 모든 청크(원문의 98%)가 선택되어 검색의 의미가 없어짐
"""
retriever = vectorstore.as_retriever(
    search_type="mmr",  # 최대 다양성 검색
    search_kwargs={"k": 3, "fetch_k": 6}  # 전체 청크(7개)보다 작게 설정
)

# 5. 프롬프트 엔지니어링
def generate_prompt(query, context):
    return f"""다음은 소득세법 비과세소득 관련 조항입니다. 문맥을 고려하여 질문에 답변하세요.

[관련 조항]
{context}

[질문]
{query}

[답변 요구사항]
- 비과세소득 유형을 계층적으로 구분하여 설명
- 각 항목별 구체적인 조건 명시
- 법조문의 항, 호, 목 번호를 포함
- 500자 이내로 간결하게 요약"""

# 검색 및 응답 생성
query = "소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?"
retrieved_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# LLM 은 위(5. Generation)에서 만든 llm 을 재사용 (temperature=0 → 앞 비교와 같은 조건)
response = llm.invoke(generate_prompt(query, context))

print('개선된 결과:')
pprint(response.content)

✓ FAISS 인덱스 저장 완료: ../db/faiss_index_openai_v1
개선된 결과:
('소득세법에서 비과세소득은 다음과 같이 구분됩니다.\n'
 '\n'
 '1. **공익신탁 소득 (제12조 제1호)**\n'
 '   - 「공익신탁법」에 따른 공익신탁의 이익.\n'
 '\n'
 '2. **사업소득 (제12조 제2호)**\n'
 '   - 가. 논ㆍ밭에서의 작물 생산 소득.\n'
 '   - 나. 1개의 주택 소유자의 주택임대소득 (기준시가 12억원 초과 주택 제외).\n'
 '   - 다. 대통령령으로 정하는 농어가부업소득.\n'
 '   - 라. 전통주 제조 소득.\n'
 '   - 마. 조림기간 5년 이상 임지의 임목 벌채 소득 (연 600만원 이하).\n'
 '   - 바. 대통령령으로 정하는 작물재배업 소득.\n'
 '   - 사. 대통령령으로 정하는 어로어업 또는 양식어업 소득.\n'
 '\n'
 '3. **근로소득 및 퇴직소득 (제12조 제3호)**\n'
 '   - 거. 국외 또는 북한지역 근로에 대한 급여.\n'
 '   - 너. 국가가 부담하는 보험료.\n'
 '   - 더. 대통령령으로 정하는 연장근로, 야간근로 급여.\n'
 '   - 러. 사내급식 등으로 제공받는 식사대 (월 20만원 이하).\n'
 '   - 머. 출산 및 6세 이하 자녀 보육 관련 급여 (월 20만원 이내).\n'
 '   - 버. 국군포로의 보수 및 퇴직일시금.\n'
 '   - 서. 대학생이 근로 대가로 받는 장학금.\n'
 '   - 어. 직무발명 보상금 (특수관계자 제외).\n'
 '\n'
 '4. **기타소득 (제12조 제5호)**\n'
 '   - 가. 보훈급여금 및 정착금.\n'
 '   - 나. 국가보안법에 따른 상금.\n'
 '   - 다. 상훈법에 따른 부상 및 상금.\n'
 '   - 라. 퇴직 후 직무발명 보상금 (특수관계자 제외).\n'
 '   - 마. 국군포로의 위로지원금.\n'
 '   - 바. 문화유산 양도로 발생하는 소득.

### Ollama 버전 vs OpenAI 버전 비교

같은 문서(`taxinfo.txt`), 같은 질문, 같은 RAG 구조에서 **임베딩과 LLM만 다르게** 실행한 두 노트북(`..._ollama`, `..._openai`)을 비교합니다.

#### 1. 구성 비교

| 항목 | Ollama 버전 | OpenAI 버전 |
|---|---|---|
| 실행 위치 | 내 PC (로컬) | OpenAI 서버 (API) |
| 임베딩 모델 | `bge-m3` (1024차원) | `text-embedding-ada-002` (1536차원) |
| 채팅 모델 | `qwen2.5:0.5b` (파라미터 약 5억 개) | `gpt-4o-mini` |
| API 키, 비용 | 필요 없음, 무료 | `OPENAI_API_KEY` 필요, 호출량만큼 과금 |
| 문서·질문의 외부 전송 | 없음 | 있음 (OpenAI로 전송됨) |
| 추가 설정 | `temperature=0`에서 반복 오류가 나서 `repeat_penalty` 필요 | 추가 설정 없이 `temperature=0` 사용 |
| 인덱스 저장 경로 | `../db/faiss_index_ollama` | `../db/faiss_index_openai` |

- 두 인덱스는 **임베딩 모델과 차원이 달라 서로 호환되지 않습니다.** 그래서 저장 경로를 분리함.

#### 2. 결과 비교 (각 노트북에 저장된 출력 기준)

| 실험 | Ollama (`qwen2.5:0.5b`) | OpenAI (`gpt-4o-mini`) |
|---|---|---|
| **context 적용** | 항목이 섞이고 글자가 깨짐 (`공익신탁`이 `공익신타피`로 출력), 소득 유형을 잘못 분류 | 제12조의 항목(공익신탁, 사업소득 `가~사`, 근로·퇴직소득, 기타소득)을 원문대로 정리 |
| **context 미적용** | 소득세법과 무관한 내용을 지어냄 (`부인의 자녀 2명 이상` 등) | 상속세·증여세, 보험금 같은 **일반적인 세금 상식**을 나열. 제12조의 실제 항목과 다름 |
| **version1** (MMR, 프롬프트 요구사항) | 저장된 출력 없음. 직접 실행했을 때는 깨진 글자와 반복된 항목이 나옴 | 요구한 구조(호·목 번호 포함, 계층 구분)를 따르고 원문의 목 표기(`거`, `너`, `더`...)까지 반영 |

**해석**
- **RAG의 효과는 두 모델 모두에서 확인됩니다.** context를 주지 않으면 두 모델 모두 제12조와 다른 내용을 답했습니다.
- **context를 주어도 답변 품질은 모델 크기에 크게 좌우됩니다.** 0.5b 소형 모델은 context를 일부 잘못 옮기고 글자가 깨졌고, `gpt-4o-mini`는 원문을 안정적으로 정리함.
- 즉 RAG는 **좋은 검색 결과를 만드는 단계**이고, 그 결과를 **정확하게 활용하는 능력은 LLM 성능**에 달려 있습니다.

#### 3. OpenAI 답변에서도 확인되는 한계

`gpt-4o-mini`의 답변도 원문과 대조하면 완벽하지 않습니다.
- **분류 오류**: 원문에서 종교인소득은 제3호(근로소득과 퇴직소득)의 한 목(`아.`)인데, context 적용 결과에서는 별도 항목(4번)으로 분류했습니다.
- **항목 누락**: 원문의 **제4호(연금소득)** 가 두 답변(context 적용, version1)에 모두 빠져 있습니다. 검색된 context에 포함되지 않았거나 모델이 생략했을 가능성이 있으며, 원인은 확인하지 못함.
- version1은 검색량을 `k=3`으로 줄여 context가 짧아졌습니다. **검색량을 줄이면 비용과 잡음은 줄지만 누락 위험은 커지므로**, 답변은 항상 원문과 대조해야 합니다.

#### 4. 선택 가이드

| 상황 | 권장 |
|---|---|
| 사내 문서, 개인정보 등 **외부로 보내면 안 되는 데이터** | Ollama (로컬). 단, `qwen2.5:1.5b` 이상 등 더 큰 모델을 권장 |
| 비용 없이 학습, 오프라인 환경 | Ollama |
| **답변 품질과 한국어 안정성**이 우선 | OpenAI |
| 호출량이 많아 비용이 부담 | 로컬 모델을 키우거나 더 저렴한 모델 검토 |

#### 5. 비교 결과를 해석할 때의 한계
- **실행 속도와 비용은 측정하지 않았습니다.** 위 표의 비용은 구조적인 차이만 적은 것입니다.
- 각 실험은 **질문 하나, 한 번 실행한 결과**라 경향을 보는 용도입니다. 정확한 평가에는 질문을 늘리고 반복 실행해야 합니다.
- 모델 크기가 크게 다르므로(약 5억 개 vs 대형 모델) "Ollama가 나쁘다"가 아니라 **같은 크기의 모델이 아닌 비교**라는 점을 감안하세요.